# Lab 24-06 · The Energy and Speed of a Short Wire

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import d2d


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Energy

*Predict* the supply energy of one rising edge.

In [2]:
g, c = d2d.ladder(50, 50, 200e-15, 50e-15)
print(f"C_tot = {c.sum()*1e15:.0f} fF; C V^2 = {c.sum()*0.25*1e15:.1f} fJ; per bit (random) {d2d.unterminated_energy(c.sum(), 0.5)*1e15:.2f} fJ")
sim = ngspice("wire.cir")
print(f"ngspice: supply energy {sim['esrc']*1e15:.2f} fJ, rise {sim['trise']*1e12:.2f} ps, 50% delay {sim['tdel']*1e12:.2f} ps")

C_tot = 300 fF; C V^2 = 75.0 fJ; per bit (random) 18.75 fJ


ngspice: supply energy 74.92 fJ, rise 43.69 ps, 50% delay 16.52 ps


## Part 2 · Speed

*Predict* whether the rise time is shorter or longer than a 32 GT/s UI (31.25 ps).

In [3]:
t = np.linspace(0, 150e-12, 600)
plt.figure()
for ron in (50, 25):
    g, c = d2d.ladder(ron, 50, 200e-15, 50e-15)
    x = d2d.step_response(g, c, 0.5)
    t10, t50, t90 = (d2d.crossing(x, 40, f * 0.5) for f in (0.1, 0.5, 0.9))
    el = d2d.elmore(ron, 50, 200e-15, 50e-15, 50e-15)
    print(f"Ron {ron} ohm: rise {(t90-t10)*1e12:.1f} ps, 50% delay {t50*1e12:.1f} ps, Elmore {el*1e12:.1f} ps")
    plt.plot(t * 1e12, [x(ti)[-1] for ti in t], label=f"driver {ron} ohm")
plt.axvline(31.25, color="0.5", ls=":")
plt.xlabel("time (ps)"); plt.ylabel("receiver voltage (V)"); plt.legend(); plt.grid(True)
plt.show()

Ron 50 ohm: rise 43.7 ps, 50% delay 16.5 ps, Elmore 22.5 ps
Ron 25 ohm: rise 28.0 ps, 50% delay 11.2 ps, Elmore 15.0 ps


## Part 3 · Density and fabrics

In [4]:
for p in (130, 55, 45, 25, 9):
    rate = 4 if p == 9 else 32
    print(f"pitch {p:3d} um at {rate} Gb/s: {d2d.bumps_per_mm2(p):8.0f} bumps/mm2, {d2d.areal_density_gbps_mm2(p, rate):7.0f} GB/s/mm2")
print(f"NVL72: {72 * 18 * 2 * 2 * 200 / 8 / 1e3:.1f} TB/s; 1 TB/s at 0.25 pJ/b: {d2d.power_w(1e12, 0.25):.1f} W")

pitch 130 um at 32 Gb/s:       59 bumps/mm2,     118 GB/s/mm2
pitch  55 um at 32 Gb/s:      331 bumps/mm2,     661 GB/s/mm2
pitch  45 um at 32 Gb/s:      494 bumps/mm2,     988 GB/s/mm2
pitch  25 um at 32 Gb/s:     1600 bumps/mm2,    3200 GB/s/mm2
pitch   9 um at 4 Gb/s:    12346 bumps/mm2,    3086 GB/s/mm2
NVL72: 129.6 TB/s; 1 TB/s at 0.25 pJ/b: 2.0 W
